In [1]:
# ===============================
# 1. Imports & Drive Mount
# ===============================
import pandas as pd
import numpy as np
from google.colab import drive

drive.mount('/content/drive')

# ===============================
# 2. Load CSV
# ===============================
file_path = "/content/drive/MyDrive/GEE_Forest_TimeSeries/HN_Forest_TimeSeries_Master.csv"
df = pd.read_csv(file_path)



Mounted at /content/drive


In [2]:
print(df.head())
print(df['year'].unique())
print(df.columns)
# Keep only one row per location-year (average if needed)
# Define feature columns
features = ['BSI', 'Forest', 'NBR', 'NDMI', 'NDVI']

# Now group properly
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
df = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()
sequence_length = 4  # 2021–2024
X_raw = []
locations = []

# Make sure duplicates are removed
df_unique = df.groupby(['latitude', 'longitude', 'year'])[features].mean().reset_index()

for (lat, lon), group in df_unique.groupby(['latitude', 'longitude']):
    group = group.sort_values('year')
    if len(group) == sequence_length:
        X_raw.append(group[features].values)
        locations.append([lat, lon])

X_raw = np.array(X_raw)       # (samples, 4, 4)
locations = np.array(locations)

print("Raw input shape:", X_raw.shape)
# =====================================================
# 1. Imports
# =====================================================
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import classification_report

# =====================================================
# 2. Create Labels (NDVI drop & BSI rise)
# =====================================================
# X_raw shape: (samples, time_steps, features)
# locations shape: (samples, 2) -> latitude, longitude

ndvi_drop = X_raw[:, 0, 0] - X_raw[:, -1, 0]   # NDVI 2021-2024
bsi_rise  = X_raw[:, -1, 3] - X_raw[:, 0, 3]   # BSI 2024-2021

# Use 85th percentile thresholds (adjust if too small)
ndvi_thr = np.percentile(ndvi_drop, 85)
bsi_thr  = np.percentile(bsi_rise, 85)

y = ((ndvi_drop > ndvi_thr) & (bsi_rise > bsi_thr)).astype(int)

print("\nLabel distribution:")
print("No deforestation:", (y == 0).sum())
print("Deforestation:", (y == 1).sum())

# =====================================================
# 3. Train-Test Split (Safe for small datasets)
# =====================================================
min_class_count = np.min([np.sum(y==0), np.sum(y==1)])
if min_class_count < 2:
    print("WARNING: Tiny dataset, skipping stratified split.")
    X_train, X_test, y_train, y_test, loc_train, loc_test = (
        X_raw, X_raw, y, y, locations, locations
    )
else:
    X_train, X_test, y_train, y_test, loc_train, loc_test = train_test_split(
        X_raw, y, locations,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)

# =====================================================
# 4. Scaling (No data leakage)
# =====================================================
scaler = MinMaxScaler()

X_train_rs = X_train.reshape(-1, X_train.shape[-1])
X_test_rs  = X_test.reshape(-1, X_test.shape[-1])

X_train = scaler.fit_transform(X_train_rs).reshape(X_train.shape)
X_test  = scaler.transform(X_test_rs).reshape(X_test.shape)

print("Scaled train shape:", X_train.shape)

# =====================================================
# 5. LSTM Model
# =====================================================
model = Sequential([
    LSTM(64, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', 'Recall', 'Precision']
)

model.summary()

# =====================================================
# 6. Train Model (Use class weighting if enough samples)
# =====================================================
if min_class_count < 2:
    print("Skipping class weighting due to tiny dataset.")
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=1,  # small batch for tiny data
        verbose=1
    )
else:
    class_weight = {0: 1, 1: 12}
    history = model.fit(
        X_train, y_train,
        validation_data=(X_test, y_test),
        epochs=20,
        batch_size=128,
        class_weight=class_weight,
        verbose=1
    )

# =====================================================
# 7. Evaluation
# =====================================================
y_prob = model.predict(X_test)
y_pred = (y_prob > 0.35).astype(int)  # recall-friendly threshold

print(classification_report(y_test, y_pred))

# =====================================================
# 8. Save Predictions (with locations)
# =====================================================
results_df = pd.DataFrame({
    'latitude': loc_test[:, 0],
    'longitude': loc_test[:, 1],
    'deforestation': y_pred.flatten()
})

print(results_df.head())

results_df.to_csv(
    '/content/drive/MyDrive/HN_Deforestation_Predictions.csv',
    index=False
)

print("✅ Saved: HN_Deforestation_Predictions.csv")

             system:index       BSI  Forest       NBR      NDMI      NDVI  \
0  00000000000000000001_0  0.094329       1  0.011236 -0.023049  0.213129   
1  00000000000000000001_0  0.094562       1 -0.020127 -0.058953  0.183084   
2  00000000000000000001_0  0.098790       1  0.011913 -0.049166  0.121263   
3  00000000000000000001_0  0.042027       1  0.095360  0.023466  0.216136   
4  00000000000000000001_0  0.046970       1  0.051586  0.012504  0.186816   

   year                                               .geo  longitude  \
0  2021  {"geodesic":false,"type":"Point","coordinates"...  77.253767   
1  2021  {"geodesic":false,"type":"Point","coordinates"...  77.253767   
2  2022  {"geodesic":false,"type":"Point","coordinates"...  77.253767   
3  2023  {"geodesic":false,"type":"Point","coordinates"...  77.253767   
4  2023  {"geodesic":false,"type":"Point","coordinates"...  77.253767   

    latitude  
0  27.840138  
1  27.840138  
2  27.840138  
3  27.840138  
4  27.840138  
[2021 20

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        17,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,033 (78.25 KB)

 Trainable params: 20,033 (78.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 202ms/step - Precision: 0.1315 - Recall: 0.8305 - accuracy: 0.2333 - loss: 1.7097 - val_Precision: 0.1376 - val_Recall: 1.0000 - val_accuracy: 0.1376 - val_loss: 0.7900
Epoch 2/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - Precision: 0.1356 - Recall: 1.0000 - accuracy: 0.1356 - loss: 1.6365 - val_Precision: 0.1376 - val_Recall: 1.0000 - val_accuracy: 0.1376 - val_loss: 0.8822
Epoch 3/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 115ms/step - Precision: 0.1356 - Recall: 1.0000 - accuracy: 0.1356 - loss: 1.6078 - val_Precision: 0.1376 - val_Recall: 1.0000 - val_accuracy: 0.1376 - val_loss: 0.9643
Epoch 4/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 106ms/step - Precision: 0.1356 - Recall: 1.0000 - accuracy: 0.1356 - loss: 1.6093 - val_Precision: 0.1376 - val_Recall: 1.0000 - val_accuracy: 0.1376 - val_loss: 1.0272
Epoch 5/20
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - Precision: 0.1356 - Recall: 1.0000 - accuracy: 0.1356 - loss: 1.6012 - val_Precision: 0.1376 - val_Recall: 1.0000 - v

In [3]:
!pip install folium
import pandas as pd
import folium
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np

# 1️⃣ Load deforestation points
df_def = pd.read_csv('/content/drive/MyDrive/HN_Deforestation_Predictions.csv')
df_deforest = df_def[df_def['deforestation']==1]

# 2️⃣ Load time-series indices
df_ts = pd.read_csv('/content/drive/MyDrive/GEE_Forest_TimeSeries/HN_Forest_TimeSeries_Master.csv')

# 3️⃣ Sort and compute changes
df_ts_sorted = df_ts.sort_values(by=['latitude','longitude','year'])

# First and last year values per point
first_year = df_ts_sorted.groupby(['latitude','longitude']).first().reset_index()
last_year  = df_ts_sorted.groupby(['latitude','longitude']).last().reset_index()

df_change = pd.merge(first_year, last_year, on=['latitude','longitude'], suffixes=('_start','_end'))

# Compute delta indices
df_change['NDVI_change'] = df_change['NDVI_end'] - df_change['NDVI_start']
df_change['NBR_change']  = df_change['NBR_end']  - df_change['NBR_start']
df_change['BSI_change']  = df_change['BSI_end']  - df_change['BSI_start']
df_change['NDMI_change'] = df_change['NDMI_end'] - df_change['NDMI_start']


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# ==============================
# THRESHOLD FUNCTION
# ==============================
def get_thresholds(series):

    series = series.dropna()

    mean = series.mean()
    std  = series.std()
    q1   = series.quantile(0.25)
    q3   = series.quantile(0.75)

    return {
        'mean': mean,
        'std': std,
        'q1': q1,
        'q3': q3,
        'lower_2std': mean - 2*std,
        'upper_2std': mean + 2*std
    }

# ==============================
# COMPUTE THRESHOLDS
# ==============================
ndvi_thresh = get_thresholds(df_change['NDVI_change'])
nbr_thresh  = get_thresholds(df_change['NBR_change'])
bsi_thresh  = get_thresholds(df_change['BSI_change'])
ndmi_thresh = get_thresholds(df_change['NDMI_change'])

print("NDVI:", ndvi_thresh)
print("NBR :", nbr_thresh)
print("BSI :", bsi_thresh)
print("NDMI:", ndmi_thresh)

# ==============================
# MERGE DATA
# ==============================
df_merged = pd.merge(
    df_deforest,
    df_change[['latitude','longitude',
               'NDVI_change','NBR_change',
               'BSI_change','NDMI_change']],
    on=['latitude','longitude'],
    how='left'
)

# ==============================
# ADAPTIVE CLASSIFIER
# ==============================
def classify_cause_adaptive(row):

    ndvi = row['NDVI_change']
    nbr  = row['NBR_change']
    bsi  = row['BSI_change']

    if pd.isna(ndvi) or pd.isna(nbr) or pd.isna(bsi):
        return "Unknown"

    if nbr < nbr_thresh['lower_2std']:
        return "Fire"

    elif ndvi < ndvi_thresh['q1'] and bsi > bsi_thresh['q3']:
        return "Logging"

    elif ndvi < ndvi_thresh['mean'] and bsi > bsi_thresh['q1']:
        return "Agriculture"

    else:
        return "Urban/Other"

# ==============================
# APPLY CLASSIFICATION
# ==============================
df_merged['cause'] = df_merged.apply(classify_cause_adaptive, axis=1)

# ==============================
# SAVE OUTPUT
# ==============================
df_merged.to_csv(
    '/content/drive/MyDrive/HN_Deforestation_Causes_Adaptive.csv',
    index=False
)

# ==============================
# SUMMARY
# ==============================
print("\nDeforestation Cause Summary:")
print(df_merged['cause'].value_counts())


import pandas as pd
import folium
from folium.plugins import MarkerCluster

# ===========================
# Haryana Map
# Approx latitude: 27–30, longitude: 74–77
# ===========================
m = folium.Map(location=[29.0, 76.0], zoom_start=7)

df = pd.read_csv('/content/drive/MyDrive/HN_Deforestation_Causes_Adaptive.csv')

print(df['cause'].value_counts())

# Define colors for causes
cause_colors = {
    'Fire': 'red',
    'Logging': 'orange',
    'Agriculture': 'yellow',
    'Urban/Other': 'gray'
}

marker_cluster = MarkerCluster().add_to(m)

for _, row in df.iterrows():
    color = cause_colors.get(row['cause'], 'blue')

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=3,
        color=color,
        fill=True,
        fill_opacity=0.7,
    ).add_to(marker_cluster)

# Save map
m.save('/content/drive/MyDrive/hn_adaptive.html')

print("✅ Haryana map saved successfully!")

NDVI: {'mean': np.float64(-0.10268894655630514), 'std': 0.11699877092951481, 'q1': np.float64(-0.1767728575), 'q3': np.float64(-0.037603392499999985), 'lower_2std': np.float64(-0.3366864884153348), 'upper_2std': np.float64(0.13130859530272448)}
NBR : {'mean': np.float64(-0.08676296781806984), 'std': 0.11252064891493527, 'q1': np.float64(-0.162552015), 'q3': np.float64(-0.0170269625), 'lower_2std': np.float64(-0.3118042656479404), 'upper_2std': np.float64(0.13827833001180068)}
BSI : {'mean': np.float64(0.04070621100660079), 'std': 0.06467034295124761, 'q1': np.float64(-0.0017871576471231755), 'q3': np.float64(0.0827098843343478), 'lower_2std': np.float64(-0.08863447489589443), 'upper_2std': np.float64(0.17004689690909602)}
NDMI: {'mean': np.float64(-0.04746908395883272), 'std': 0.07960706181223683, 'q1': np.float64(-0.10047794), 'q3': np.float64(0.00581921249999999), 'lower_2std': np.float64(-0.20668320758330638), 'upper_2std': np.float64(0.11174503966564094)}

Deforestation Cause Summa

In [9]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================
df_def = pd.read_csv(
    '/content/drive/MyDrive/HN_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()
print("Total deforestation samples:", len(df_deforest))

# ==============================
# LOAD CAUSE DATA
# ==============================
df_cause = pd.read_csv(
    '/content/drive/MyDrive/HN_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())

# Ensure latitude/longitude are float to avoid merge issues
df_deforest['latitude'] = df_deforest['latitude'].astype(float)
df_deforest['longitude'] = df_deforest['longitude'].astype(float)
df_cause['latitude'] = df_cause['latitude'].astype(float)
df_cause['longitude'] = df_cause['longitude'].astype(float)

# ==============================
# LOAD HARYANA DISTRICTS
# ==============================
hn = gpd.read_file('/content/drive/MyDrive/hn.geojson')

# CRS SAFETY
if hn.crs is None:
    hn.set_crs("EPSG:4326", inplace=True)

hn = hn.to_crs("EPSG:4326")
hn["state"] = "Haryana"

print(hn.head())

# ==============================
# MERGE CAUSE DATA
# ==============================
df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())

# ==============================
# CREATE POINT GEOMETRY
# ==============================
geometry = [Point(xy) for xy in zip(df_deforest['longitude'], df_deforest['latitude'])]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 218
    latitude  longitude  deforestation
0  29.658328  76.253044              0
1  28.978304  75.900904              1
2  29.844280  74.630686              1
3  29.434648  75.582901              0
4  28.530044  76.563861              0
Total deforestation samples: 132
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  28.978304  75.900904              1    -0.072324   -0.038433    0.020704   
1  29.844280  74.630686              1    -0.054333    0.054987    0.002339   
2  29.755346  75.834429              1    -0.100819   -0.083126    0.019946   
3  29.610718  75.893718              1    -0.027624   -0.062056    0.001533   
4  29.660125  75.637698              1    -0.046464   -0.115865   -0.010982   

   NDMI_change        cause  
0    -0.012613  Urban/Other  
1    -0.012334  Urban/Other  
2     0.023168  Urban/Other  
3     0.012691  Urban/Other  
4     0.026073  Urban/Other  
  REMARKS_2 Country State_Name State_Code  Dist_Name Dist_C

In [10]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# ==============================
# LOAD DEFORESTATION PREDICTIONS
# ==============================
df_def = pd.read_csv(
    '/content/drive/MyDrive/HN_Deforestation_Predictions.csv'
)

print("Total samples:", len(df_def))
print(df_def.head())

# Filter only deforestation = 1
df_deforest = df_def[df_def['deforestation'] == 1].copy()
print("Total deforestation samples:", len(df_deforest))

# ==============================
# LOAD CAUSE DATA
# ==============================
df_cause = pd.read_csv(
    '/content/drive/MyDrive/HN_Deforestation_Causes_Adaptive.csv'
)

print(df_cause.head())

# Ensure latitude/longitude are float to avoid merge issues
df_deforest['latitude'] = df_deforest['latitude'].astype(float)
df_deforest['longitude'] = df_deforest['longitude'].astype(float)
df_cause['latitude'] = df_cause['latitude'].astype(float)
df_cause['longitude'] = df_cause['longitude'].astype(float)

# ==============================
# LOAD HARYANA DISTRICTS
# ==============================
hn = gpd.read_file('/content/drive/MyDrive/hn.geojson')

# CRS SAFETY
if hn.crs is None:
    hn.set_crs("EPSG:4326", inplace=True)

hn = hn.to_crs("EPSG:4326")
hn["state"] = "Haryana"

print(hn.head())

# ==============================
# MERGE CAUSE DATA
# ==============================
df_deforest = df_deforest.merge(
    df_cause[['latitude', 'longitude', 'cause']],
    on=['latitude', 'longitude'],
    how='left'
)

print(df_deforest.head())

# ==============================
# CREATE POINT GEOMETRY
# ==============================
geometry = [Point(xy) for xy in zip(df_deforest['longitude'], df_deforest['latitude'])]

gdf_points = gpd.GeoDataFrame(
    df_deforest,
    geometry=geometry,
    crs="EPSG:4326"
)

print(gdf_points.head())

Total samples: 218
    latitude  longitude  deforestation
0  29.658328  76.253044              0
1  28.978304  75.900904              1
2  29.844280  74.630686              1
3  29.434648  75.582901              0
4  28.530044  76.563861              0
Total deforestation samples: 132
    latitude  longitude  deforestation  NDVI_change  NBR_change  BSI_change  \
0  28.978304  75.900904              1    -0.072324   -0.038433    0.020704   
1  29.844280  74.630686              1    -0.054333    0.054987    0.002339   
2  29.755346  75.834429              1    -0.100819   -0.083126    0.019946   
3  29.610718  75.893718              1    -0.027624   -0.062056    0.001533   
4  29.660125  75.637698              1    -0.046464   -0.115865   -0.010982   

   NDMI_change        cause  
0    -0.012613  Urban/Other  
1    -0.012334  Urban/Other  
2     0.023168  Urban/Other  
3     0.012691  Urban/Other  
4     0.026073  Urban/Other  
  REMARKS_2 Country State_Name State_Code  Dist_Name Dist_C

In [12]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
import numpy as np

# =====================================================
# STEP 1: Load Haryana District Boundaries
# =====================================================
hn = gpd.read_file('/content/drive/MyDrive/hn.geojson')

# CRS safety
if hn.crs is None:
    hn.set_crs(epsg=4326, inplace=True)

hn = hn.to_crs(epsg=4326)
hn["state"] = "Haryana"

gdf_districts = hn.copy()

# =====================================================
# STEP 2: Load Deforestation Predictions
# =====================================================
df = pd.read_csv("/content/drive/MyDrive/HN_Deforestation_Predictions.csv")

print("Total samples:", len(df))

# Ensure afforestation column exists
if 'afforestation' not in df.columns:
    df['afforestation'] = 0

# =====================================================
# STEP 3: Convert to GeoDataFrame
# =====================================================
gdf_points = gpd.GeoDataFrame(
    df,
    geometry=[Point(xy) for xy in zip(df.longitude, df.latitude)],
    crs="EPSG:4326"
)

# =====================================================
# STEP 4: Spatial Join
# =====================================================
points_with_district = gpd.sjoin(
    gdf_points,
    gdf_districts,
    how="left",
    predicate="within"
)

# =====================================================
# STEP 5: District Statistics
# =====================================================

district_total = (
    points_with_district
    .groupby("Dist_Name")
    .size()
    .reset_index(name="total_samples")
)

district_deforestation = (
    points_with_district[points_with_district["deforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="deforestation_samples")
)

district_afforestation = (
    points_with_district[points_with_district["afforestation"] == 1]
    .groupby("Dist_Name")
    .size()
    .reset_index(name="afforestation_samples")
)

# =====================================================
# STEP 6: Merge Statistics
# =====================================================
gdf_districts = gdf_districts.merge(
    district_total,
    on="Dist_Name",
    how="left"
)

gdf_districts = gdf_districts.merge(
    district_deforestation,
    on="Dist_Name",
    how="left"
)

gdf_districts = gdf_districts.merge(
    district_afforestation,
    on="Dist_Name",
    how="left"
)

gdf_districts.fillna(0, inplace=True)

# =====================================================
# STEP 7: Area + Rate Calculations
# =====================================================

PIXEL_AREA_SQKM = 0.0001
PIXEL_AREA_HA = 0.01

gdf_districts["deforestation_rate_%"] = (
    gdf_districts["deforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["deforestation_area_sqkm"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["deforestation_area_ha"] = (
    gdf_districts["deforestation_samples"] * PIXEL_AREA_HA
)

gdf_districts["afforestation_rate_%"] = (
    gdf_districts["afforestation_samples"] /
    gdf_districts["total_samples"] * 100
).fillna(0)

gdf_districts["afforestation_area_sqkm"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_SQKM
)

gdf_districts["afforestation_area_ha"] = (
    gdf_districts["afforestation_samples"] * PIXEL_AREA_HA
)

# =====================================================
# STEP 8: Create Haryana Map
# =====================================================
m = folium.Map(location=[29.0, 76.0], zoom_start=7)

# =====================================================
# STEP 9: Choropleth Map
# =====================================================

min_val = gdf_districts["deforestation_samples"].min()
max_val = gdf_districts["deforestation_samples"].max()

if max_val == min_val:
    bins = [min_val, max_val + 1]
else:
    bins = np.linspace(min_val, max_val, num=6).tolist()

folium.Choropleth(
    geo_data=gdf_districts.to_json(),
    data=gdf_districts,
    columns=["Dist_Name", "deforestation_samples"],
    key_on="feature.properties.Dist_Name",
    fill_color="YlOrRd",
    bins=bins,
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Deforestation Samples (Haryana)"
).add_to(m)

# =====================================================
# STEP 10: Tooltips
# =====================================================

folium.GeoJson(
    gdf_districts,
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "Dist_Name",
            "deforestation_samples",
            "deforestation_rate_%",
            "deforestation_area_sqkm",
            "deforestation_area_ha",
            "afforestation_samples",
            "afforestation_rate_%",
            "afforestation_area_sqkm",
            "afforestation_area_ha"
        ],
        aliases=[
            "District:",
            "Deforestation samples:",
            "Deforestation rate (%):",
            "Deforestation Area (sq.km):",
            "Deforestation Area (ha):",
            "Afforestation samples:",
            "Afforestation rate (%):",
            "Afforestation Area (sq.km):",
            "Afforestation Area (ha):"
        ],
        localize=True
    )
).add_to(m)

# =====================================================
# STEP 11: Alert Popup
# =====================================================

state_def = gdf_districts["deforestation_samples"].sum()

top3 = (
    gdf_districts.sort_values(
        by="deforestation_samples",
        ascending=False
    ).head(3)
)

top_districts_html = ""

for _, row in top3.iterrows():
    top_districts_html += f"• {row['Dist_Name']} ({int(row['deforestation_samples'])})<br>"

popup_html = f"""
<b>🌳 Haryana Afforestation Alert</b><br><br>
Total Deforestation Points: <b>{int(state_def)}</b><br><br>
High Impact Districts:<br>
{top_districts_html}<br>
🌱 Immediate afforestation drives recommended.
"""

folium.Marker(
    location=[29.0, 76.0],
    popup=popup_html,
    icon=folium.Icon(color="green")
).add_to(m)

# =====================================================
# STEP 12: Save Map
# =====================================================

folium.LayerControl().add_to(m)

m.save("/content/drive/MyDrive/hn_forest.html")

print("✅ Haryana map saved successfully!")

Total samples: 218


/tmp/ipykernel_202/1903094018.py:97: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  gdf_districts.fillna(0, inplace=True)


✅ Haryana map saved successfully!
